# Agricultural Chemical Use — PPT-Aligned EDA

**Project:** *Agricultural Chemical Use: Identifying Crop Systems for Reduced Chemical Dependence*  
**Source:** USDA NASS 2025 Cotton and Potatoes Quick Stats pesticide-use and pest-management exports.

This notebook deliberately follows the descriptive analysis in the final presentation: (1) compare percent of planted acres treated for herbicides, insecticides, fungicides, and other chemicals; (2) summarize selected potato pest-management practices; and (3) document the actual Quick Stats exports and data-quality limitations. Additional QA is included, but the main findings stay aligned with the presentation.

**Important:** The four category percentages below are the rounded crop-level headline estimates published in the USDA NASS 2025 highlights and used in the presentation. They are not calculated by summing active-ingredient rows. Category percentages can overlap and must not be added together. This is descriptive analysis; no state-level regression or causal effect is claimed.

## 1. Imports and file loading

Place this notebook in the same folder as the four CSV files listed below. Keep the original Quick Stats exports unchanged.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path('.')
FILES = {
    'cotton_pesticide': DATA_DIR / 'cotton_pesticide_use_2025_quickstats.csv',
    'potato_pesticide': DATA_DIR / 'potato_pesticide_use_2025_quickstats.csv',
    'cotton_practices': DATA_DIR / 'cotton_pest_management_2025_quickstats.csv',
    'potato_practices': DATA_DIR / 'potato_pest_management_2025_quickstats.csv',
}
missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Missing CSV file(s): ' + ', '.join(missing) + '. Put the four CSV files next to this notebook.')

raw = {name: pd.read_csv(path, dtype=str, keep_default_na=False) for name, path in FILES.items()}
for name, df in raw.items():
    print(f'{name}: {df.shape[0]:,} rows × {df.shape[1]} columns')


## 2. Reproduce the presentation’s crop comparison

Primary measure: **percent of planted acres treated**. It answers how widespread reported use is. Rate and applications per acre describe intensity, while total pounds describe overall scale; they are supporting measures and should not be blended into one score.

In [ ]:
# Rounded crop-level estimates reported in USDA NASS 2025 Cotton and Potato Highlights
# These values match the presentation's Results / Appendix comparison table.
coverage = pd.DataFrame({
    'Category': ['Herbicides', 'Insecticides', 'Fungicides', 'Other chemicals'],
    'Cotton (%)': [88, 39, 1, 54],
    'Potatoes (%)': [90, 92, 96, 29],
})
display(coverage)

fig, ax = plt.subplots(figsize=(10, 5.8))
y = np.arange(len(coverage))
h = 0.34
b1 = ax.barh(y - h/2, coverage['Cotton (%)'], height=h, label='Cotton')
b2 = ax.barh(y + h/2, coverage['Potatoes (%)'], height=h, label='Potatoes')
ax.set_yticks(y, coverage['Category'])
ax.invert_yaxis()
ax.set_xlim(0, 108)
ax.set_xlabel('Percent of planted acres treated (%)')
ax.set_title('2025 pesticide-category coverage: cotton vs. potatoes')
ax.grid(axis='x', alpha=0.25)
ax.set_axisbelow(True)
ax.bar_label(b1, fmt='%.0f%%', padding=3)
ax.bar_label(b2, fmt='%.0f%%', padding=3)
ax.legend(frameon=False, ncols=2, loc='lower right')
fig.tight_layout()
plt.show()

coverage['Potatoes minus cotton (percentage points)'] = coverage['Potatoes (%)'] - coverage['Cotton (%)']
display(coverage)


### Interpretation
- Potato coverage is high for herbicides (90%), insecticides (92%), and fungicides (96%).
- Cotton coverage is high for herbicides (88%) and “other chemicals” (54%); insecticide coverage is 39%, and fungicide coverage is 1%.
- The largest displayed difference is fungicides: potato coverage is 95 percentage points above cotton.
- These are coverage estimates, **not** chemical mass, toxicity, risk, or proof that a lower-chemical alternative will maintain yield. The two crop surveys cover different sets of states.

## 3. Reproduce the potato pest-management context

These selected practice percentages match the presentation. They describe reported use and do not establish that the practices caused lower pesticide use.

In [ ]:
potato_practices = pd.DataFrame({
    'Practice': ['Scouting', 'Equipment cleaning', 'Crop rotation', 'Ground cover / physical barriers'],
    'Percent of planted acres (%)': [99, 86, 75, 79],
})
display(potato_practices)

plot_data = potato_practices.sort_values('Percent of planted acres (%)')
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(plot_data['Practice'], plot_data['Percent of planted acres (%)'])
ax.set_xlim(0, 108)
ax.set_xlabel('Percent of planted acres (%)')
ax.set_title('Selected reported potato pest-management practices (2025)')
ax.grid(axis='x', alpha=0.25)
ax.set_axisbelow(True)
ax.bar_label(bars, fmt='%.0f%%', padding=3)
fig.tight_layout()
plt.show()


## 4. Validate the four Quick Stats exports

The raw files include state records and, for pesticide use, multi-state aggregate records. Practice files in this export are multi-state totals. This section checks dimensions, geography, domains, and value/suppression patterns without converting suppressed estimates into zero.

In [ ]:
def find_value_columns(df):
    return [c for c in df.columns if '<b>VALUE</b>' in c]

def quality_summary(name, df):
    value_cols = find_value_columns(df)
    out = {
        'file_role': name,
        'rows': len(df),
        'columns': len(df.columns),
        'geography_levels': ', '.join(sorted(df['Geo Level'].replace('', np.nan).dropna().unique())) if 'Geo Level' in df else 'not available',
        'states_present': int(df['State'].replace('', np.nan).dropna().nunique()) if 'State' in df else 0,
        'domains': int(df['Domain'].replace('', np.nan).dropna().nunique()) if 'Domain' in df else 0,
        'value_columns': len(value_cols),
    }
    for col in value_cols:
        s = df[col].astype(str).str.strip()
        out[col.replace(' -  <b>VALUE</b>', '').replace('  -  <b>VALUE</b>', '') + ' | blank'] = int(s.eq('').sum())
        out[col.replace(' -  <b>VALUE</b>', '').replace('  -  <b>VALUE</b>', '') + ' | coded/suppressed'] = int(s.str.contains(r'\([A-Z]\)|^[A-Z]$', regex=True).sum())
    return out

quality = pd.DataFrame([quality_summary(name, df) for name, df in raw.items()])
display(quality.T)

for name in ['cotton_pesticide', 'potato_pesticide']:
    df = raw[name]
    print(f'\n{name}: geographic level counts')
    display(df['Geo Level'].replace('', '(blank)').value_counts().rename_axis('Geo Level').to_frame('rows'))
    print(f'{name}: domain counts')
    display(df['Domain'].replace('', '(blank)').value_counts().rename_axis('Domain').to_frame('rows'))


### Reading USDA codes and uncertainty
- Preserve the original value strings. USDA symbols such as `(D)` indicate withheld/suppressed data; they are not zero. Other symbols may indicate unavailable or non-publishable estimates. Consult the USDA symbol and data-item definitions linked in the project source page.
- Coefficients of variation (CVs) are available in some pesticide-use columns. Review them before using an estimate to rank states.
- Do not combine multi-state aggregate rows with individual state rows. Do not rank states until the specific measure, suppression, independent state count, and estimate precision have been validated.

## 5. Explore selected potato practices from the Quick Stats export

The presentation’s four practice percentages above are the exact headline values used in the slides. This optional table reads the detailed practice export and shows matching practice labels where available. It is a cross-check, not a replacement for the slide’s published headline table.

In [ ]:
def value_column(df):
    cols = find_value_columns(df)
    if not cols:
        raise ValueError('No VALUE column found')
    # Practice exports contain one planted-acre percentage VALUE column.
    return next((c for c in cols if 'PCT OF AREA PLANTED' in c), cols[0])

pdf = raw['potato_practices'].copy()
pv = value_column(pdf)
pdf['value_raw'] = pdf[pv].astype(str).str.strip()
pdf['value_numeric'] = pd.to_numeric(pdf['value_raw'].str.replace(',', '', regex=False), errors='coerce')

# Keep labels readable but preserve the full source label in the table.
practice_terms = {
    'SCOUTED - ESTABLISHED PROCESS USED': 'Scouted — established process',
    'EQUIPMENT & IMPLEMENTS CLEANED AFTER FIELD WORK TO REDUCE SPREAD OF PESTS': 'Equipment cleaning',
    'ROTATED CROPS DURING PAST 3 YEARS': 'Crop rotation',
    'GROUND COVERS, MULCHES, OR OTHER PHYSICAL BARRIERS MAINTAINED': 'Ground cover / physical barriers',
}
rows = []
for _, row in pdf.iterrows():
    label = str(row.get('Domain Category', ''))
    upper = label.upper()
    for term, short in practice_terms.items():
        if term in upper:
            rows.append({'Practice': short, 'Source label': label, 'Value as published': row['value_raw'], 'Numeric value (if reported)': row['value_numeric']})
            break
display(pd.DataFrame(rows))


## 6. Conclusions and limits (aligned to the presentation)

**Recommendation:** prioritize research on an integrated lower-chemical approach in potato systems, and evaluate cotton’s more focused weed-management pathway separately. The survey indicates where use is widespread; it does not establish that a substitute works.

**What this EDA supports**
- A crop-level descriptive comparison of percent of planted acres treated.
- A focused description of selected reported potato pest-management practices.
- Quality checks on the real Quick Stats exports and documentation of suppressed estimates.

**What this EDA does not claim**
- No state is named as the “winner” or top pilot location.
- No state-level regression has been fitted. A model is only a candidate after validating independent units, missing/suppressed estimates, and precision/CVs; with a small state sample, descriptive profiles may be more defensible.
- No causal reduction, pesticide hazard ranking, yield protection, or profitability effect is inferred. A prospective field pilot is needed to test those outcomes.

## Sources
- USDA NASS, [2025 Cotton Chemical Use Highlights](https://www.nass.usda.gov/Surveys/Guide_to_NASS_Surveys/Chemical_Use/2025_Cotton_Potatoes/ChemHighlights2025CottonFINAL.pdf)
- USDA NASS, [2025 Potato Chemical Use Highlights](https://www.nass.usda.gov/Surveys/Guide_to_NASS_Surveys/Chemical_Use/2025_Cotton_Potatoes/ChemHighlights2025PotatoFINAL.pdf)
- USDA NASS, [2025 Cotton & Potatoes Quick Stats queries](https://www.nass.usda.gov/Data_and_Statistics/Pre-Defined_Queries/2025_Cotton_Potatoes/index.php)
- USDA NASS, [Field Crops Methodology & Quality Measures](https://www.nass.usda.gov/Publications/Methodology_and_Data_Quality/Agricultural_Chemical_Usage_-_Field_Crops/index.php)
